1 - What does a SavedModel contain? How do you inspect its content?

The saved model contains a metagraph of the model, containing its computational graph, de type and shape of inputs and outputs, definitions of functions, a name, training operations. An initialization function and a serve function.

To inspect its content there is a saved_model_cli tool

2 - When should you use TF Serving? What are its main features? What are some tools you can use to deploy it?

Sustain high loads of traffic, serve multiple versions of your models and watch a model repository to automatically deploy the lastest version. 

A use of it is to serve a model with docker. 

Query a model by restAPI or by gRPC



3 - How do you deploy a model across multiple TF Serve instances?

Install the google loud platform library, set project id, location and then create a new vertex ai model with the path to the model, and the url of the docker container that will run the model. Then an endpoint needs to be created and then the replicas parameters tells the service how many replicas can be used.

4 - When should you use the gRPC API instead of the API Rest to call a model server by TF Serving ? 

The Rest Api works with JSON, so it has to represent floats as strings and then convert them to floats to receive the input, and in the output step convert them back. If the number has 15 characters that is 120 bits against the 32 of a float. 

So for large inputs, the best practice is to use protobuf and gRPC

5 - What are the different ways TFLites reduces a model's size to to make it run on mobile devices?

It can downgrade float precision to 16 bits

It can adapt the model to use integers instead of floats, scaling the values of the model. This is called post-training-quantization. It reduces the apps size, but not improves performance or memory usage.

It can reduce the metagraph information of the model, like removing the training operations

It also optimizes computations that can be simplified to a single one

It can fuse operations, like folding batch normalization layers into the previous layer's addition and multiplication operations

It can compress the savedModels to the flatBuffers format, it is similar to protobufs but for models. So the model can go into RAM without any preprocessing, this saves energy and compute.



6 - What is quantization aware training?

Is a technique for when a model's weights are quantized, if performance drops too much, then the model should be trained using this technique that adds noise during training, so the model then ignores the noise inserted by the quantization.

7 - What are model parallelism and data parallelism? Why is the latter generally recommended?

## Model Parallelism
Model parallelism is a technique to train models in several devices in order to shorten the time used in training. It consits in splitting the architechture of the model and assigning each part to a different device. The problem with this approach is that un a fully connected layer for example, computing each layer in a different device just adds waiting time. And vertically dividing the model is really complex because a halve always uses values from the other halve.

RNNs can be benefited of this approach because each cell can run in a separate device taking the most out of performance, but the time that a cell wastes in transfering data to another leaves us with a slightly better performance.

## Data Parallelism
Data parallelism consists in training multiple models in several devices, each with a replica of the model. To calculate gradients, the means of the gradients of all the runs is needed to update the weights of all the models and then continue the process.

Mirrored strategy
All replicas have the same values and are updated with the same values on every GPU. It is the most efficient technique.

Centralized Parameters strategy
It is based on storing the model parameters outside the GPUs that are performing calculations. In a distributed setup, you may place the parameters on a parameter server called CPU-servers, whose role is to host and update the parameters. The update of the weights can be syncronous of asynchronous. When synchronous, the parameter server has to wait for all the GPUs to finish the computations, then pass the results to the optimizer, update the weights and send them back to the GPUs with the gradients applied (that can really load the data transfer bandwiths). Asynchronous updates means that when a GPU has finished computing gradients they are inmediately applied in the parameters server, without calculating a mean or waiting for replicas. The synchronization of parameters is also avoided, but the parameters are copied to each device after updating. This technique is hard to use because the learning never converges.

8 - When training a model at scale: What distributions strategies can you use? How do you choose between them?

MultiWorkerMirroredStrategy for a mirrored strategy approach explained above
ParameterServerStrategy for a centralized parameters approach